In [1]:
# Install required packages

!pip install -q langchain langchain-openai langgraph langchain-tavily

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 kB 2.8 MB/s eta 0:00:00


In [2]:
# Import the libraries needed for ModelRank

import os
import json
import time
import requests

from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import TypedDict

from google.colab import userdata
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain_tavily import TavilySearch
from langgraph.graph import StateGraph, END

In [3]:
# Load API keys from Colab Secrets

os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")

In [4]:
# Define the language model used by the agents

model = ChatOpenAI(
    model="google/gemini-3.8-flash",
    temperature=0,
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY"),
)

# Agent 1 - Planner Agent

In [5]:
# Planner Agent - Tools

tavily_search = TavilySearch(max_results=5)


@tool
def search_models(query: str) -> str:
    """Search the web for current information about suitable LLMs."""

    results = tavily_search.invoke(
        {
            "query": query
        }
    )

    return json.dumps(results, ensure_ascii=False)


@tool
def save_evaluation_plan(
    project_name: str,
    project_type: str,
    project_goal: str,
    evaluation_criteria: list[str],
    candidate_models: list[dict],
    notes: str = ""
) -> str:
    """Save the evaluation plan as a JSON file."""

    plan = {
        "project_name": project_name,
        "project_type": project_type,
        "project_goal": project_goal,
        "evaluation_criteria": evaluation_criteria,
        "candidate_models": candidate_models,
        "notes": notes
    }

    with open("evaluation_plan.json", "w", encoding="utf-8") as file:
        json.dump(plan, file, indent=2, ensure_ascii=False)

    return "Evaluation plan saved to evaluation_plan.json"

In [6]:
# Planner Agent - Prompt

PLANNER_PROMPT = """
You are the Planner Agent in ModelRank.

Your job is to understand the user's AI project and create a clear evaluation plan.

You should identify:
- The project name
- The project type
- The main project goal
- The most important evaluation criteria
- Suitable candidate models to compare
- Any important notes

First, understand the user's project and evaluation needs.

Evaluation criteria must come only from the user's request.
Do not add evaluation criteria based on search results.

Then, use the search_models tool to search for current information
about suitable LLMs based on the user's project and evaluation criteria,
including their exact OpenRouter model IDs.

Use the search_models tool only once unless the first search returns
insufficient results.

Use the search results only to select suitable candidate models.

Each candidate model must contain exactly:
- name
- provider
- model_id

The model_id must be the exact OpenRouter API model identifier
in the format:
provider/model-name

For example:
anthropic/claude-haiku-4.5

Do not include prefixes such as:
openrouter:

The name, provider, and model_id must refer to the exact same model.

Before selecting a candidate model, verify that its model_id matches
the exact model name and provider.

Do not guess, infer, or invent model IDs.

Select distinct candidate models with specific model names.
Avoid duplicate, overlapping, or generic model names.

Prefer candidate models from different providers when possible,
as long as they are relevant to the user's requirements.

Select only candidate models that are supported by the search results.

Do not claim that candidate models meet the user's requirements
before they are benchmarked.

Do not state or imply that ModelRank has already benchmarked,
tested, ranked, or validated the candidate models.

Notes may briefly explain why the models were selected based on
external search information, but must not present those claims
as ModelRank benchmark results.

Focus only on creating the evaluation plan.
Do not generate test cases or execute model evaluations.

Do not invent specific numbers, constraints, requirements,
domain examples, use cases, or assumptions that were not explicitly
mentioned by the user.

Do not invent model names.
Candidate models must come from the search results.

Keep notes brief and only include information necessary
to explain the model selection.

After selecting the candidate models, always use the save_evaluation_plan tool
to save the plan as evaluation_plan.json.

After saving the plan, respond with exactly:
"Evaluation plan saved to evaluation_plan.json"

Do not summarize the plan.
Do not explain the selected models.
Do not ask questions.
Do not suggest or perform any next steps.

Keep the plan simple, relevant, and focused on the user's project.
"""

In [7]:
# Planner Agent - Create Agent

planner_agent = create_agent(
    model=model,
    tools=[search_models, save_evaluation_plan],
    system_prompt=PLANNER_PROMPT,
)

In [8]:
# Planner Agent - Test

query = """
I am building an AI customer support chatbot for an e-commerce website.
The chatbot should provide clear and accurate answers,
follow instructions well,
respond quickly,
and keep API costs low.
I want to compare multiple LLMs and choose the best one for production.
"""

result = planner_agent.invoke({
    "messages": [HumanMessage(content=query)]
})

for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================


I am building an AI customer support chatbot for an e-commerce website.
The chatbot should provide clear and accurate answers,
follow instructions well,
respond quickly,
and keep API costs low.
I want to compare multiple LLMs and choose the best one for production.

================================== Ai Message ==================================
Tool Calls:
  search_models (call_372889)
 Call ID: call_372889
  Args:
    query: best fast low cost instruction following models openrouter
================================= Tool Message =================================
Name: search_models

{"query": "best fast low cost instruction following models openrouter", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://openrouter.ai/collections/discounted-models", "title": "Discounted AI Models on OpenRouter | OpenRouter", "content": "Benchmarked by Artificial Analysis, MiniMax-M2 

# Agent 2 - Benchmark Agent

In [9]:
# Benchmark Agent - Tools

@tool
def load_evaluation_plan() -> str:
    """Load the evaluation plan created by the Planner Agent."""

    with open("evaluation_plan.json", "r", encoding="utf-8") as file:
        plan = json.load(file)

    return json.dumps(plan, ensure_ascii=False)


@tool
def save_benchmark(
    project_name: str,
    test_cases: list[dict]
) -> str:
    """Save the benchmark test cases as a JSON file."""

    benchmark = {
        "project_name": project_name,
        "test_cases": test_cases
    }

    with open("benchmark.json", "w", encoding="utf-8") as file:
        json.dump(benchmark, file, indent=2, ensure_ascii=False)

    return "Benchmark saved to benchmark.json"

In [10]:
# Benchmark Agent - Prompt

BENCHMARK_PROMPT = """
You are the Benchmark Agent in ModelRank.

Your job is to create a compact evaluation benchmark for the user's project.

First, use the load_evaluation_plan tool to read the evaluation plan created by the Planner Agent.

Use the project goal and evaluation criteria from the evaluation plan to create relevant test cases.

Each test case must use exactly these keys:
- id
- criterion
- prompt
- expected_behavior

Do not use alternative or misspelled field names.

Create test cases only for criteria that require model responses,
such as accuracy, clarity, reasoning, or instruction following.

Do not create test cases for latency or API cost.
Those will be measured during model execution.

Make every test case self-contained whenever the model needs factual,
policy, product, account, or business information to answer correctly.

For accuracy-related test cases, include all facts, policies, rules,
or context needed to determine the correct answer inside the prompt itself.

For instruction-following test cases, if the task also requires answering
a factual or domain-specific question, include the necessary information
inside the prompt so the model does not need to invent an answer.

Do not invent external facts, company policies, prices, dates,
shipping times, return rules, warranty periods, store hours,
account procedures, or other information that was not provided.

Do not infer rules from missing information.
If a detail is not stated in the prompt, do not treat it as a requirement
or exception.

Avoid ambiguous test cases that allow multiple conflicting answers.
Each test case should have a clear and verifiable expected result.

The expected_behavior must be based only on information contained
inside the test case prompt.

Expected behavior must only judge facts and instructions that are
explicitly stated in the prompt.

If the prompt does not provide enough information to infer an additional
detail, the expected_behavior should make clear that the model should not
invent or infer that detail.

Make expected behaviors specific, clear, and verifiable.

Before saving the benchmark, verify that every expected_behavior
is logically consistent with the facts, rules, and instructions
stated in its prompt.

Keep the benchmark small and focused.
Create exactly 3 test cases.

Avoid duplicate or unnecessary test cases.

Do not evaluate models.
Do not select the best model.
Do not change the candidate models.

Do not invent requirements that are not supported by the evaluation plan.

Do not print, display, summarize, or explain the benchmark before saving it.

Create the test cases internally, then immediately call the save_benchmark tool.

Do not output the test cases in the assistant message.

After creating the test cases, always use the save_benchmark tool
to save them as benchmark.json.

After saving the benchmark, respond with exactly:
"Benchmark saved to benchmark.json"

Do not summarize the benchmark.
Do not ask questions.
Do not suggest or perform any next steps.
"""

In [11]:
# Benchmark Agent - Create Agent

benchmark_agent = create_agent(
    model=model,
    tools=[load_evaluation_plan, save_benchmark],
    system_prompt=BENCHMARK_PROMPT,
)

In [12]:
# Benchmark Agent - Test

result = benchmark_agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="Create the benchmark using the saved evaluation plan."
            )
        ]
    }
)

for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

Create the benchmark using the saved evaluation plan.
================================== Ai Message ==================================
Tool Calls:
  load_evaluation_plan (call_791129)
 Call ID: call_791129
  Args:
================================= Tool Message =================================
Name: load_evaluation_plan

{"project_name": "E-Commerce Customer Support Chatbot", "project_type": "Customer Support Chatbot", "project_goal": "Compare multiple LLMs and choose the best one for production to power an AI customer support chatbot for an e-commerce website.", "evaluation_criteria": ["Clear and accurate answers", "Instruction following", "Response speed", "API cost"], "candidate_models": [{"model_id": "meta-llama/llama-3.3-70b-instruct", "provider": "meta-llama", "name": "Llama 3.3 70B Instruct"}, {"model_id": "mistralai/mistral-small-3.1-24b-instruct", "provider": "mistralai", "name": "Mistral Small 3

# Agent 3 - Runner Agent

In [13]:
# Runner Agent - Tools


@tool
def load_runner_inputs() -> str:
    """Load the evaluation plan and benchmark."""

    with open("evaluation_plan.json", "r", encoding="utf-8") as file:
        plan = json.load(file)

    with open("benchmark.json", "r", encoding="utf-8") as file:
        benchmark = json.load(file)

    inputs = {
        "evaluation_plan": plan,
        "benchmark": benchmark
    }

    return json.dumps(inputs, ensure_ascii=False)


@tool
def run_models_parallel(runner_inputs: str) -> str:
    """Run all candidate models on the same benchmark in parallel."""

    inputs = json.loads(runner_inputs)

    plan = inputs["evaluation_plan"]
    benchmark = inputs["benchmark"]

    candidate_models = plan["candidate_models"]
    test_cases = benchmark["test_cases"]

    api_key = os.environ.get("OPENROUTER_API_KEY")
    url = "https://openrouter.ai/api/v1/chat/completions"

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }

    def run_model(model_info):
        model_results = []

        for test_case in test_cases:

            payload = {
                "model": model_info["model_id"],
                "messages": [
                    {
                        "role": "user",
                        "content": test_case["prompt"]
                    }
                ],
                "temperature": 0,
                "usage": {
                    "include": True
                }
            }

            start_time = time.perf_counter()

            try:
                response = requests.post(
                    url,
                    headers=headers,
                    json=payload,
                    timeout=120
                )

                response.raise_for_status()

                elapsed_time = time.perf_counter() - start_time
                data = response.json()

                output = data["choices"][0]["message"]["content"]
                usage = data.get("usage", {})

                model_results.append(
                    {
                        "test_case_id": test_case["id"],
                        "criterion": test_case["criterion"],
                        "expected_behavior": test_case["expected_behavior"],
                        "output": output,
                        "latency_seconds": round(elapsed_time, 3),
                        "input_tokens": usage.get("prompt_tokens"),
                        "output_tokens": usage.get("completion_tokens"),
                        "total_tokens": usage.get("total_tokens"),
                        "cost_usd": usage.get("cost"),
                        "error": None
                    }
                )

            except Exception as error:
                elapsed_time = time.perf_counter() - start_time

                model_results.append(
                    {
                        "test_case_id": test_case["id"],
                        "criterion": test_case["criterion"],
                        "expected_behavior": test_case["expected_behavior"],
                        "output": None,
                        "latency_seconds": round(elapsed_time, 3),
                        "input_tokens": None,
                        "output_tokens": None,
                        "total_tokens": None,
                        "cost_usd": None,
                        "error": str(error)
                    }
                )

        return {
            "name": model_info["name"],
            "provider": model_info["provider"],
            "model_id": model_info["model_id"],
            "results": model_results
        }

    all_results = []

    with ThreadPoolExecutor(
        max_workers=len(candidate_models)
    ) as executor:

        futures = [
            executor.submit(
                run_model,
                model_info
            )
            for model_info in candidate_models
        ]

        for future in as_completed(futures):
            all_results.append(
                future.result()
            )

    results = {
        "project_name": plan["project_name"],
        "models": all_results
    }

    return json.dumps(results, ensure_ascii=False)


@tool
def save_runner_results(results_json: str) -> str:
    """Calculate model summaries and save the runner results."""

    runner_results = json.loads(results_json)

    for model in runner_results["models"]:

        successful_results = [
            result
            for result in model["results"]
            if result["error"] is None
        ]

        failed_results = [
            result
            for result in model["results"]
            if result["error"] is not None
        ]

        latencies = [
            result["latency_seconds"]
            for result in successful_results
        ]

        costs = [
            result["cost_usd"]
            for result in successful_results
            if result["cost_usd"] is not None
        ]

        input_tokens = [
            result["input_tokens"]
            for result in successful_results
            if result["input_tokens"] is not None
        ]

        output_tokens = [
            result["output_tokens"]
            for result in successful_results
            if result["output_tokens"] is not None
        ]

        model["summary"] = {
            "average_latency_seconds": (
                round(sum(latencies) / len(latencies), 3)
                if latencies
                else None
            ),
            "total_cost_usd": (
                round(sum(costs), 8)
                if costs
                else None
            ),
            "total_input_tokens": sum(input_tokens),
            "total_output_tokens": sum(output_tokens),
            "successful_tests": len(successful_results),
            "failed_tests": len(failed_results)
        }

    with open("runner_results.json", "w", encoding="utf-8") as file:
        json.dump(
            runner_results,
            file,
            indent=2,
            ensure_ascii=False
        )

    return "Runner results saved to runner_results.json"

In [14]:
# Runner Agent - Prompt

RUNNER_PROMPT = """
You are the Runner Agent in ModelRank.

Your job is to execute the benchmark on all candidate models
and save the runtime results.

First, use the load_runner_inputs tool to load:
- The evaluation plan
- The benchmark test cases

Then, pass the output of load_runner_inputs directly to
the run_models_parallel tool.

Use the run_models_parallel tool to execute all candidate models
on the same benchmark.

The candidate models must be executed in parallel.

Then, pass the output of run_models_parallel directly to
the save_runner_results tool.

The save_runner_results tool must calculate the model summaries
and save the final results to runner_results.json.

The same benchmark must be used for every candidate model.

Do not modify:
- The candidate models
- The benchmark test cases
- The expected behaviors
- The evaluation criteria

Do not evaluate the quality of model responses.
Do not decide whether a response passed or failed the expected behavior.
Do not select the best model.
Do not rank the models.
Do not make recommendations.

Your job is only to execute the benchmark and collect runtime data.

For every test case, collect:
- Test case ID
- Evaluation criterion
- Expected behavior
- Model output
- Response latency
- Input token usage
- Output token usage
- Total token usage
- API cost when available
- Any execution error

For every model, calculate:
- Average response latency
- Total API cost
- Total input tokens
- Total output tokens
- Number of successful executions
- Number of failed executions

Always follow this tool order:

1. load_runner_inputs
2. run_models_parallel
3. save_runner_results

After save_runner_results completes successfully, respond with exactly:
"Benchmark completed and saved to runner_results.json"

Do not print or summarize the benchmark results.
Do not explain model performance.
Do not ask questions.
Do not suggest or perform any next steps.
"""

In [15]:
# Runner Agent - Create Agent

runner_agent = create_agent(
    model=model,
    tools=[
        load_runner_inputs,
        run_models_parallel,
        save_runner_results
    ],
    system_prompt=RUNNER_PROMPT,
)

In [16]:
# Runner Agent - Test

result = runner_agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="Run the benchmark using the saved evaluation plan and benchmark."
            )
        ]
    }
)

for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

Run the benchmark using the saved evaluation plan and benchmark.
================================== Ai Message ==================================
Tool Calls:
  load_runner_inputs (call_375329)
 Call ID: call_375329
  Args:
================================= Tool Message =================================
Name: load_runner_inputs

{"evaluation_plan": {"project_name": "E-Commerce Customer Support Chatbot", "project_type": "Customer Support Chatbot", "project_goal": "Compare multiple LLMs and choose the best one for production to power an AI customer support chatbot for an e-commerce website.", "evaluation_criteria": ["Clear and accurate answers", "Instruction following", "Response speed", "API cost"], "candidate_models": [{"model_id": "meta-llama/llama-3.3-70b-instruct", "provider": "meta-llama", "name": "Llama 3.3 70B Instruct"}, {"model_id": "mistralai/mistral-small-3.1-24b-instruct", "provider": "mistralai

# Agent 4 - Judge Agent

In [17]:
# Judge Agent - Tools


@tool
def load_runner_results() -> str:
    """Load the benchmark results produced by the Runner Agent."""

    with open("runner_results.json", "r", encoding="utf-8") as file:
        results = json.load(file)

    return json.dumps(results, ensure_ascii=False)


@tool
def save_final_decision(
    selected_model: dict,
    model_rankings: list[dict],
    decision_reason: str,
    retry_required: bool,
    retry_reason: str = ""
) -> str:
    """Save the Judge Agent's final model selection decision."""

    decision = {
        "selected_model": selected_model,
        "model_rankings": model_rankings,
        "decision_reason": decision_reason,
        "retry_required": retry_required,
        "retry_reason": retry_reason
    }

    with open("final_decision.json", "w", encoding="utf-8") as file:
        json.dump(
            decision,
            file,
            indent=2,
            ensure_ascii=False
        )

    return "Final decision saved to final_decision.json"

In [18]:
# Judge Agent - Prompt

JUDGE_PROMPT = """
You are the Judge Agent in ModelRank.

Your job is to evaluate the benchmark results and select the best model
for the user's project.

First, use the load_runner_results tool to read the benchmark results.

For each model, evaluate:
- How well the model output matches the expected_behavior
- How accurately the model follows the provided facts and policies
- How well the model follows explicit instructions and formatting requirements
- Average response latency
- Total API cost
- Any execution errors

Use the evaluation criteria from the benchmark results and project context.

Do not rely on model reputation, provider reputation, or outside knowledge.
Judge only from the benchmark results produced by ModelRank.

For response quality, compare each model output directly against
the corresponding expected_behavior.

Do not treat a successful API execution as a successful benchmark result.
A model can execute successfully but still fail the expected behavior.

Consider both response quality and runtime performance.

Prioritize correctness and instruction following over small differences
in latency or cost.

If two models have similar response quality, prefer the model with
lower latency and lower API cost.

Rank all candidate models from best to worst.

Select exactly one best model when at least one candidate performs
well enough for the project.

Set retry_required to true only if all candidate models perform poorly,
fail important benchmark requirements, or are not suitable for the project.

If retry_required is true:
- Explain clearly why the current models are insufficient
- Do not select a model as the final recommendation

If retry_required is false:
- Select the best-performing model
- Explain the decision using benchmark evidence
- Avoid absolute claims such as "perfect", "flawless", or "fully correct"
  unless they are directly and objectively verified by the benchmark
- Prefer evidence-based wording such as "best overall compliance",
  "strongest benchmark performance", or "highest observed compliance"

Do not modify benchmark results.
Do not rerun models.
Do not create new test cases.
Do not search for new models.

After evaluating the results, always use the save_final_decision tool.

The selected_model must include:
- name
- provider
- model_id

Each item in model_rankings should include:
- rank
- name
- provider
- model_id
- quality_assessment
- average_latency_seconds
- total_cost_usd
- reason

Keep the decision_reason concise and based only on benchmark evidence.

After saving the decision, respond with exactly:
"Final decision saved to final_decision.json"

Do not summarize the decision in the assistant message.
Do not ask questions.
Do not suggest or perform any next steps.
"""

In [19]:
# Judge Agent - Create Agent

judge_agent = create_agent(
    model=model,
    tools=[
        load_runner_results,
        save_final_decision
    ],
    system_prompt=JUDGE_PROMPT,
)

In [20]:
# Judge Agent - Test

result = judge_agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="Evaluate the saved runner results and select the best model."
            )
        ]
    }
)

for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

Evaluate the saved runner results and select the best model.
================================== Ai Message ==================================
Tool Calls:
  load_runner_results (call_1231833)
 Call ID: call_1231833
  Args:
================================= Tool Message =================================
Name: load_runner_results

{"models": [{"model_id": "mistralai/mistral-small-3.1-24b-instruct", "name": "Mistral Small 3.1 24B Instruct", "provider": "mistralai", "results": [{"cost_usd": 0.000109008, "criterion": "Clear and accurate answers", "error": null, "expected_behavior": "The response must state that the shirt is eligible for return because it is within 30 days, unworn with tags attached, and not a final sale item. It must explicitly clarify that returning to the original payment method (credit card) will incur a $5.99 deducted shipping fee, rather than a 100% full refund without deduction. It should

# LangGraph

In [21]:
# LangGraph - State


class ModelRankState(TypedDict):
    user_request: str
    retry_count: int
    retry_required: bool
    selected_model: dict | None

In [22]:
# LangGraph - Nodes


def planner_node(state: ModelRankState):
    if state["retry_count"] > 0:
        planner_message = f"""
The previous candidate models were not good enough.

Select a new set of candidate models for the same user request.

Do not select the same candidate models used in the previous attempt.

Original user request:
{state["user_request"]}
"""
    else:
        planner_message = state["user_request"]

    planner_agent.invoke(
        {
            "messages": [
                HumanMessage(
                    content=planner_message
                )
            ]
        }
    )

    return state


def benchmark_node(state: ModelRankState):
    benchmark_agent.invoke(
        {
            "messages": [
                HumanMessage(
                    content="Create the benchmark using the saved evaluation plan."
                )
            ]
        }
    )

    return state


def runner_node(state: ModelRankState):
    runner_agent.invoke(
        {
            "messages": [
                HumanMessage(
                    content="Run the benchmark using the saved evaluation plan and benchmark."
                )
            ]
        }
    )

    return state


def judge_node(state: ModelRankState):
    judge_agent.invoke(
        {
            "messages": [
                HumanMessage(
                    content="Evaluate the saved runner results and select the best model."
                )
            ]
        }
    )

    with open("final_decision.json", "r", encoding="utf-8") as file:
        decision = json.load(file)

    retry_count = state["retry_count"]

    if decision["retry_required"]:
        retry_count += 1

    return {
        **state,
        "retry_required": decision["retry_required"],
        "retry_count": retry_count,
        "selected_model": decision["selected_model"]
    }

In [23]:
# LangGraph - Routing


MAX_RETRIES = 2


def route_after_planner(state: ModelRankState):
    if state["retry_count"] > 0:
        return "runner"

    return "benchmark"


def route_after_judge(state: ModelRankState):
    if (
        state["retry_required"]
        and state["retry_count"] < MAX_RETRIES
    ):
        return "planner"

    return "end"

In [24]:
# LangGraph - Workflow


workflow = StateGraph(ModelRankState)

workflow.add_node("planner", planner_node)
workflow.add_node("benchmark", benchmark_node)
workflow.add_node("runner", runner_node)
workflow.add_node("judge", judge_node)

workflow.set_entry_point("planner")

workflow.add_conditional_edges(
    "planner",
    route_after_planner,
    {
        "benchmark": "benchmark",
        "runner": "runner"
    }
)

workflow.add_edge(
    "benchmark",
    "runner"
)

workflow.add_edge(
    "runner",
    "judge"
)

workflow.add_conditional_edges(
    "judge",
    route_after_judge,
    {
        "planner": "planner",
        "end": END
    }
)

model_fit_graph = workflow.compile()

In [25]:
# LangGraph - End-to-End Test


initial_state = {
    "user_request": """
I am building an AI customer support chatbot for an e-commerce website.

The chatbot should provide clear and accurate answers,
follow instructions well,
respond quickly,
and keep API costs low.

I want to compare multiple LLMs and choose the best one for production.
""",
    "retry_count": 0,
    "retry_required": False,
    "selected_model": None
}

final_state = model_fit_graph.invoke(initial_state)

print(final_state)

{'user_request': '\nI am building an AI customer support chatbot for an e-commerce website.\n\nThe chatbot should provide clear and accurate answers,\nfollow instructions well,\nrespond quickly,\nand keep API costs low.\n\nI want to compare multiple LLMs and choose the best one for production.\n', 'retry_count': 0, 'retry_required': False, 'selected_model': {'provider': 'meta-llama', 'name': 'Llama 4 Scout', 'model_id': 'meta-llama/llama-4-scout'}}


In [26]:
# LangGraph - Retry Routing Test


test_state = {
    "user_request": "Test request",
    "retry_count": 1,
    "retry_required": True,
    "selected_model": None
}

print(route_after_judge(test_state))

planner


In [27]:
# LangGraph - Retry Limit Test


test_state = {
    "user_request": "Test request",
    "retry_count": 2,
    "retry_required": True,
    "selected_model": None
}

print(route_after_judge(test_state))

end
